# Week 3 - Data Contract

**Student:** Zahed Shaikh

**Repository:** zahed-flyrank-ml

**Notebook:** w03_data_contract.ipynb

Track: Machine Learning

## 1. Data Contract

### What one row means

One row represents the daily performance of a single content page for one client.

### Tables used

- fact_content_daily_performance
- dim_content

### Time Window

March 2026 (month = 2026-03)

### Prediction Target

Predict whether a content page will receive high organic clicks.

### Deliberately Excluded

Client metadata and future performance data.

In [18]:
!pip -q install duckdb huggingface_hub pyarrow

In [19]:
import duckdb
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token Loaded:", HF_TOKEN is not None)

Token Loaded: True


In [20]:
con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
);
""")

repo = "hf://datasets/FlyRank/internship-warehouse"

print("Connected")

Connected


# 2. Verification Queries

In [21]:
grain = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{repo}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

grain

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      9841378 │
└──────────────┘

In [22]:
date_span = con.sql(f"""
SELECT
MIN(report_date),
MAX(report_date),
COUNT(*)
FROM read_parquet('{repo}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

date_span

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┬──────────────┐
│ min(report_date) │ max(report_date) │ count_star() │
│       date       │       date       │    int64     │
├──────────────────┼──────────────────┼──────────────┤
│ 2026-03-01       │ 2026-03-31       │      9841378 │
└──────────────────┴──────────────────┴──────────────┘

In [23]:
availability = con.sql(f"""
SELECT
COUNT(*) AS available_rows
FROM read_parquet('{repo}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
""")

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘

# 3. Feature Engineering

In [24]:
features = con.sql(f"""
SELECT

report_date,
gsc_clicks,
gsc_impressions,
gsc_sum_position,
sessions_ai

FROM read_parquet('{repo}/fact_content_daily_performance/month=2026-03/*.parquet')

LIMIT 100
""")

features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬─────────────────┬──────────────────┬─────────────┐
│ report_date │ gsc_clicks │ gsc_impressions │ gsc_sum_position │ sessions_ai │
│    date     │   int64    │      int64      │      int64       │    int64    │
├─────────────┼────────────┼─────────────────┼──────────────────┼─────────────┤
│ 2026-03-01  │          0 │              20 │               67 │        NULL │
│ 2026-03-01  │          0 │               1 │                0 │        NULL │
│ 2026-03-01  │          1 │             125 │              616 │        NULL │
│ 2026-03-01  │          0 │               7 │               28 │        NULL │
│ 2026-03-01  │          0 │              11 │               25 │        NULL │
│ 2026-03-01  │          1 │             239 │             1756 │        NULL │
│ 2026-03-01  │          0 │             191 │             1496 │        NULL │
│ 2026-03-01  │          0 │              55 │              180 │        NULL │
│ 2026-03-01  │          0 │            

## Feature Availability

| Feature | Available at decision time? |
|----------|----------------------------|
| Clicks | Yes, historical value |
| Impressions | Yes |
| CTR | Yes |
| Position | Yes |
| Report Date | Yes |

# 4. Leakage Experiment

In [25]:
leak = con.sql(f"""
SELECT

gsc_clicks,
gsc_impressions,

CASE
    WHEN gsc_clicks > 100 THEN 1
    ELSE 0
END AS label

FROM read_parquet('{repo}/fact_content_daily_performance/month=2026-03/*.parquet')

LIMIT 100
""")

leak

┌────────────┬─────────────────┬───────┐
│ gsc_clicks │ gsc_impressions │ label │
│   int64    │      int64      │ int32 │
├────────────┼─────────────────┼───────┤
│          0 │              20 │     0 │
│          0 │               1 │     0 │
│          1 │             125 │     0 │
│          0 │               7 │     0 │
│          0 │              11 │     0 │
│          1 │             239 │     0 │
│          0 │             191 │     0 │
│          0 │              55 │     0 │
│          0 │              77 │     0 │
│          0 │               2 │     0 │
│          · │               · │     · │
│          · │               · │     · │
│          · │               · │     · │
│          0 │               0 │     0 │
│          0 │               2 │     0 │
│          0 │               2 │     0 │
│          0 │               5 │     0 │
│          0 │               2 │     0 │
│          0 │               4 │     0 │
│          0 │               0 │     0 │
│          0 │  

The label above was intentionally created from the target variable (clicks).

Using it as a feature would leak future information into the model, producing unrealistically high performance.

The leaked feature is removed before model training.

# 5. Limitation

This notebook only analyzes one month of historical data (March 2026). Results may not generalize across other months or clients.

# 6. Self Check

✅ Contract completed

✅ Three verification queries executed

✅ Five features identified

✅ Leakage experiment demonstrated

✅ Limitation documented